# Handling Categorical Data

Machine Learning models work only with **numerical data**. If a dataset contains text values (e.g., Male/Female, Red/Blue/Green), we must convert them into numbers. This process is called **Encoding**.

There are two common encoding techniques:

- **Label Encoding**
- **One-Hot Encoding**

---

## 1. Label Encoding

Label Encoding assigns a **unique integer** to each category.

### Example

| Color | Encoded |
|--------|----------|
| Red | 2 |
| Blue | 0 |
| Green | 1 |

### When to Use

- Ordinal data (categories with an order)
- Example: Small < Medium < Large





In [2]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

df = pd.DataFrame({
    "Size": ["Small", "Medium", "Large", "Medium", "Small"]
})

encoder = LabelEncoder()

df["Size_Encoded"] = encoder.fit_transform(df["Size"])

print(df)

     Size  Size_Encoded
0   Small             2
1  Medium             1
2   Large             0
3  Medium             1
4   Small             2


## 2. One-Hot Encoding

One-Hot Encoding creates a **new binary column** for each category.

### Example

Original:

| Color |
|--------|
| Red |
| Blue |
| Green |

After Encoding:

| Red | Blue | Green |
|-----|------|-------|
|1|0|0|
|0|1|0|
|0|0|1|

### When to Use

- Nominal data (categories without any order)
- Example: Colors, Countries, Cities


In [3]:
import pandas as pd

df = pd.DataFrame({
    "Color": ["Red", "Blue", "Green", "Red"]
})

encoded_df = pd.get_dummies(df, columns=["Color"])

print(encoded_df)

   Color_Blue  Color_Green  Color_Red
0       False        False       True
1        True        False      False
2       False         True      False
3       False        False       True


## Label Encoding vs One-Hot Encoding

| Label Encoding | One-Hot Encoding |
|----------------|------------------|
| Assigns integers | Creates multiple binary columns |
| Best for ordinal data | Best for nominal data |
| Uses one column | Uses multiple columns |

---


# Feature Scaling

Feature Scaling is the process of bringing all numerical features to a **similar scale** so that no feature dominates another due to its larger values.

Example:

| Age | Salary |
|------|---------|
|20|30000|
|30|60000|

Since salary values are much larger than age values, many algorithms perform better after scaling.

There are two common scaling techniques:

- Standardization
- Normalization

---

## 1. Standardization (Z-Score Scaling)

Standardization transforms data so that:

- Mean = 0
- Standard Deviation = 1

### Formula

\[
z=\frac{x-\mu}{\sigma}
\]

### When to Use

- Logistic Regression
- KNN
- SVM
- PCA
- K-Means


In [1]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

df = pd.DataFrame({
    "Age": [20, 25, 35, 45, 60]
})

scaler = StandardScaler()

df["Age_Standardized"] = scaler.fit_transform(df[["Age"]])

print(df)

   Age  Age_Standardized
0   20         -1.184446
1   25         -0.836080
2   35         -0.139347
3   45          0.557386
4   60          1.602486


## 2. Normalization (Min-Max Scaling)

Normalization rescales the data between **0 and 1**.

### Formula

\[
x'=\frac{x-min}{max-min}
\]

### When to Use

- Neural Networks
- Image Data
- Deep Learning


In [4]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

df = pd.DataFrame({
    "Age": [20, 25, 35, 45, 60]
})

scaler = MinMaxScaler()

df["Age_Normalized"] = scaler.fit_transform(df[["Age"]])

print(df)

   Age  Age_Normalized
0   20           0.000
1   25           0.125
2   35           0.375
3   45           0.625
4   60           1.000


## Standardization vs Normalization

| Standardization | Normalization |
|-----------------|---------------|
| Mean = 0 | Values between 0 and 1 |
| Standard Deviation = 1 | Fixed range |
| No fixed output range | Output range is 0–1 |
| Used for KNN, SVM, PCA | Used for Neural Networks |

---

# Data Leakage
## What is Data Leakage?

**Data leakage** occurs when information from the **test dataset** is unintentionally used during the training process.

As a result, the model appears to perform very well during testing, but performs poorly on new, unseen data.

---

## Why is Data Leakage a Problem?

The purpose of the **test set** is to evaluate how well the model performs on **unseen data**.

If the model (or any preprocessing step) has already learned something from the test data, the evaluation is no longer fair.

This leads to:

- Unrealistically high accuracy
- Poor real-world performance
- An unreliable machine learning model

---

# How Can Scaling Cause Data Leakage?

Many beginners think that scaling simply changes the values.

In reality, a scaler **learns statistics from the data** before scaling it.

For example:

- **StandardScaler** learns:
  - Mean
  - Standard Deviation

- **MinMaxScaler** learns:
  - Minimum value
  - Maximum value

If these statistics are calculated using the **entire dataset**, then information from the test set has already been used.

This is called **data leakage**.

---

## ❌ Wrong Approach

Scale the data **before** splitting.

```python
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

scaler = StandardScaler()

# Wrong: Learns from the entire dataset
X_scaled = scaler.fit_transform(X)

# Split after scaling
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)
```

### Why is this wrong?

The scaler calculates the **mean** and **standard deviation** using **both the training and test data**.

So, the training process has indirectly received information about the test set.

This is **data leakage**.

---

## ✅ Correct Approach

Split the data first.

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split first
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

# Learn statistics only from training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaling to the test data
X_test_scaled = scaler.transform(X_test)
```

### Why is this correct?

- `fit()` learns the mean and standard deviation **only from the training data**.
- `transform()` applies the same scaling to the test data.
- The test data remains completely unseen during training.

---

# Understanding `fit()` and `transform()`

### `fit()`

Learns information from the data.

Example:

```python
scaler.fit(X_train)
```

For `StandardScaler`, it learns:

- Mean
- Standard Deviation

---

### `transform()`

Uses the learned statistics to scale the data.

```python
X_test_scaled = scaler.transform(X_test)
```

No new statistics are calculated.

The same mean and standard deviation learned from the training data are used.

---

### `fit_transform()`

A shortcut for:

```python
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
```

---

# Best Practice Workflow

```text
Dataset
    │
    ▼
Train-Test Split
    │
    ▼
Fit Preprocessing on Training Data
    │
    ▼
Transform Training Data
    │
    ▼
Transform Test Data
    │
    ▼
Train Model
    │
    ▼
Evaluate Model
```

---

# Golden Rule

> **If a preprocessing technique learns anything from the data (mean, standard deviation, minimum, maximum, categories, principal components, etc.), always fit it only on the training data.**

Examples include:

- StandardScaler
- MinMaxScaler
- SimpleImputer
- PCA
- Feature Selection
- OneHotEncoder
- LabelEncoder (when applicable)

---



## Best Practice

Always perform **Train-Test Split first**, then fit the scaler only on the training data.

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the training data
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

# Scale the test data using the same scaler
X_test = scaler.transform(X_test)
```

This prevents **data leakage** and ensures fair model evaluation.